# DataONE / CIB sampling events → FCOs → sample-prep context
Dataset: *California Insect and Soil Biodiversity Sampling, Stillwater Sciences (May 2023–Aug 2025)*,
California Institute for Biodiversity, CC0 1.0 (DataONE series `urn:uuid:8d8187ba-…`, latest `urn:uuid:4929e888-…`).

Each sampling event becomes an FCO (personal fields dropped). Its sample type, habitat and soil moisture set
prep expectations before imaging: extraction method, finest useful mesh, flow, and body types to protect.
The Sample ID can travel with the bag as BugPicker's `collection_event_slot`.

**Swap data in the next cell**: any DataONE PID (a newer version is followed automatically) or a local CSV.

In [ ]:
# ---- swap your data here ----
PID = "urn:uuid:8d8187ba-683e-4c14-b5a5-c00655e4e852"
OUT = "/Volumes/magicBLACKbox/baebugs/datasets/dataone_cib"   # raw data stays on the drive, not in git
LOCAL_CSV = None        # skip download and use this file
SAMPLE_ID = None        # a bag's Sample ID, e.g. "20230524-132157-CDK"; None = pick an insect sample
REFRESH = False

In [ ]:
import sys, os, json, glob
sys.path.insert(0, "..")
import pandas as pd, matplotlib.pyplot as plt
from baebugs.dataone_adapter import pull_dataone, load_events, event_fco, events_for_bag, summary
from baebugs.essig_adapter import specimen_context, vision_prompt_context
csv = LOCAL_CSV
if not csv:
    if REFRESH or not glob.glob(f"{OUT}/*.csv"):
        m = pull_dataone(PID, OUT); print("latest:", m["latest"], "|", [f["file"] for f in m["files"]])
    csv = sorted(glob.glob(f"{OUT}/*.csv"))[-1]
df = load_events(csv)
print(json.dumps(summary(df), indent=1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.sample_type.value_counts().plot.barh(ax=ax[0], title="sample types")
pd.crosstab(df.habitat_type, df.sample_type).plot.barh(stacked=True, ax=ax[1], title="habitat x sample type", legend=True)
plt.tight_layout()
geo = df.dropna(subset=["latitude", "longitude"])
plt.figure(figsize=(5, 6)); 
for st, g in geo.groupby("sample_type"): plt.scatter(g.longitude, g.latitude, s=6, label=st)
plt.legend(fontsize=7); plt.title("sampling events (California)"); plt.xlabel("lon"); plt.ylabel("lat");

## One bag: sampling-event FCO → prep hint → specimen context for the vision model

In [ ]:
sid = SAMPLE_ID or df[df.sample_type.str.startswith("insect", na=False)].sample_id.iloc[0]
evs = events_for_bag(df, sid)
for e in evs:
    print(e.kind, e.id[:16], "|", e.fields.get("sample_type"), "|", e.fields.get("habitat_type"), "|", e.fields.get("soil_moisture"))
    print("prep hint:", json.dumps(e.fields.get("prep_hint"), indent=1))
ctx = specimen_context(*evs)
print(vision_prompt_context(ctx))

## Prep hints across the whole dataset (what the sieve lane should expect)

In [ ]:
fcos = [event_fco(r) for r in df.to_dict("records")]
rows = [dict(sample_type=f.fields.get("sample_type"), extraction=(f.fields.get("prep_hint") or {}).get("extraction"),
             mesh_floor_mm=(f.fields.get("prep_hint") or {}).get("mesh_floor_mm"),
             warning=(f.fields.get("prep_hint") or {}).get("warning")) for f in fcos]
hints = pd.DataFrame(rows)
print(hints.groupby(["sample_type", "extraction", "mesh_floor_mm"], dropna=False).size().to_string())
print("\nsaturated/flooded samples flagged for clogging:", hints.warning.notna().sum())